# CIFAR-100 Multi-Exit Vision Transformer

Benchmarking study on CIFAR-100 (100 classes, 32×32 images) comparing three baseline architectures against a custom multi-exit Vision Transformer with energy-based early exit routing. All models trained from scratch on a Kaggle T4x2 GPU without data augmentation. The primary goal is to explore adaptive inference depth — allowing easy samples to exit the network early, saving compute while preserving accuracy.

**Dataset:** CIFAR-100 — 50,000 train / 10,000 test — batch size 64  
**Hardware:** NVIDIA Tesla T4x2 (Kaggle)  
**Framework:** PyTorch 2.x  
**Optimizer:** Adam / AdamW with early stopping (patience = 10)

## Models

### 1. SimpleCNN (baseline)
Two conv blocks (`3→32→64`) with MaxPool, followed by a fully connected head (`4096→256→100`).  
Lightweight at 0.07M parameters and 3.51M MACs. Stopped at epoch 17 due to overfitting. **Best accuracy: 39.62%**

### 2. ResNet-18
Standard ResNet-18 with the final FC replaced by `Linear(512→100)`. Trained from scratch — no pretrained weights.  
Residual connections allow deeper effective learning without degradation. Stopped at epoch 18. **Best accuracy: 44.68%**

### 3. MiniTransformer
Custom ViT-style model. 4×4 patch embedding → CLS token → 2 TransformerEncoder layers (d=64, 4 heads, FFN=128) → classification head.  
Trained with AdamW + label smoothing. Ran all 50 epochs and was still improving. **Best accuracy: 31.61%**

### 4. MultiExitTransformer (proposed)
Same backbone as MiniTransformer extended to 3 layers, with an independent classification head after each layer.  
At inference, an energy score (`-logsumexp(logits)`) gates early exit — if the model is confident enough, it skips remaining layers.  
Trained with weighted multi-exit loss: `0.3 × L1 + 0.3 × L2 + 0.4 × L3`. **Best accuracy: 32.27%**

## Multi-Exit Threshold Results

Evaluated across six confidence thresholds. Lower (more negative) threshold = harder to exit early = more compute used.

| Threshold | Accuracy (%) | Avg Exit Layer | Compute Saved (%) |
|-----------|-------------|----------------|-------------------|
| -6        | 32.27       | 3.00           | 0.00              |
| -5        | 28.56       | 1.65           | 45.01             |
| -4        | 25.07       | 1.00           | 66.67             |
| -3        | 25.07       | 1.00           | 66.67             |
| -2        | 25.07       | 1.00           | 66.67             |
| -1        | 25.07       | 1.00           | 66.67             |

**Key takeaway:** Threshold `-5` is the only viable operating point — 45% compute savings at a 3.71% accuracy cost.  
Below `-4`, all samples exit at layer 1 regardless of difficulty, indicating early exit head miscalibration.  
Suggested fix: temperature scaling or knowledge distillation to better calibrate intermediate head confidence.

**Imports + Device**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


**Loading Data**

In [ ]:
 transform = transforms.Compose([
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform
)

testset = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False
)

100%|██████████| 169M/169M [00:06<00:00, 27.6MB/s] 


In [ ]:
images, labels = next(iter(trainloader))

print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Sample labels:", labels[:10])

Image shape: torch.Size([64, 3, 32, 32])
Label shape: torch.Size([64])
Sample labels: tensor([85, 36, 71, 85, 37,  3, 25, 72, 67, 83])


**Simple CNN**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 100)
        )

    def forward(self, x):
        return self.net(x)

model = SimpleCNN().to(device)

**Training + Eval**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch():
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(trainloader)


def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
EPOCHS = 50
PATIENCE = 10

best_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):

    loss = train_one_epoch()
    acc = evaluate()

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        patience_counter = 0

        torch.save(model.state_dict(), "best_simplecnn.pth")
        print("Best model updated")

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered")
        print(f"Best Accuracy: {best_acc:.2f}%")
        break


Epoch 1
Loss: 3.6089
Accuracy: 22.55%
Best model updated

Epoch 2
Loss: 2.9300
Accuracy: 29.53%
Best model updated

Epoch 3
Loss: 2.6003
Accuracy: 31.63%
Best model updated

Epoch 4
Loss: 2.3610
Accuracy: 36.35%
Best model updated

Epoch 5
Loss: 2.1602
Accuracy: 37.20%
Best model updated

Epoch 6
Loss: 1.9905
Accuracy: 36.47%

Epoch 7
Loss: 1.8311
Accuracy: 39.62%
Best model updated

Epoch 8
Loss: 1.6805
Accuracy: 39.43%

Epoch 9
Loss: 1.5393
Accuracy: 39.44%

Epoch 10
Loss: 1.4029
Accuracy: 38.61%

Epoch 11
Loss: 1.2668
Accuracy: 38.76%

Epoch 12
Loss: 1.1400
Accuracy: 38.34%

Epoch 13
Loss: 1.0308
Accuracy: 37.78%

Epoch 14
Loss: 0.9169
Accuracy: 37.72%

Epoch 15
Loss: 0.8111
Accuracy: 37.12%

Epoch 16
Loss: 0.7050
Accuracy: 37.37%

Epoch 17
Loss: 0.6279
Accuracy: 35.97%

Early stopping triggered
Best Accuracy: 39.62%


**ResNet-18**

In [ ]:
import torchvision.models as models

model = models.resnet18(pretrained=False)

model.fc = nn.Linear(model.fc.in_features, 100)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

**Training ResNet**

In [ ]:
EPOCHS = 50
PATIENCE = 10

best_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    loss = train_one_epoch()
    acc = evaluate()

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_resnet.pth")
        print("Best model updated")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered")
        print(f"Best Accuracy: {best_acc:.2f}%")
        break


Epoch 1
Loss: 3.5733
Accuracy: 23.74%
Best model updated

Epoch 2
Loss: 2.8368
Accuracy: 27.75%
Best model updated

Epoch 3
Loss: 2.4345
Accuracy: 33.79%
Best model updated

Epoch 4
Loss: 2.1474
Accuracy: 34.12%
Best model updated

Epoch 5
Loss: 1.9004
Accuracy: 39.98%
Best model updated

Epoch 6
Loss: 1.6652
Accuracy: 37.46%

Epoch 7
Loss: 1.4279
Accuracy: 39.97%

Epoch 8
Loss: 1.1899
Accuracy: 44.68%
Best model updated

Epoch 9
Loss: 0.9539
Accuracy: 44.19%

Epoch 10
Loss: 0.7435
Accuracy: 43.28%

Epoch 11
Loss: 0.5617
Accuracy: 44.34%

Epoch 12
Loss: 0.4447
Accuracy: 43.73%

Epoch 13
Loss: 0.3615
Accuracy: 44.54%

Epoch 14
Loss: 0.3232
Accuracy: 44.20%

Epoch 15
Loss: 0.2628
Accuracy: 41.78%

Epoch 16
Loss: 0.2518
Accuracy: 43.41%

Epoch 17
Loss: 0.2320
Accuracy: 44.42%

Epoch 18
Loss: 0.1897
Accuracy: 42.46%

Early stopping triggered
Best Accuracy: 44.68%


**Mini Transformer**

In [ ]:
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.embed_dim = 64

        # Flatten image into patches (8x8 patches)
        self.patch_embed = nn.Conv2d(3, self.embed_dim, kernel_size=4, stride=4)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 65, self.embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.norm = nn.LayerNorm(self.embed_dim)
        self.head = nn.Linear(self.embed_dim, 100)

    def forward(self, x):
        B = x.size(0)

        # Patch embedding
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)

        # CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls, x), dim=1)

        # Positional embedding
        x = x + self.pos_embed

        # Transformer
        x = self.transformer(x)

        # Classification
        x = self.norm(x[:, 0])
        return self.head(x)

In [ ]:
model = MiniTransformer().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

**Training mini transformer**

In [ ]:
EPOCHS = 50
PATIENCE = 10

best_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    loss = train_one_epoch()
    acc = evaluate()

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        patience_counter = 0
        torch.save(model.state_dict(), "mini_transformer.pth")
        print("Best model updated")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered")
        print(f"Best Accuracy: {best_acc:.2f}%")
        break


Epoch 1
Loss: 4.4373
Accuracy: 6.04%
Best model updated

Epoch 2
Loss: 4.1233
Accuracy: 11.23%
Best model updated

Epoch 3
Loss: 3.9499
Accuracy: 13.89%
Best model updated

Epoch 4
Loss: 3.8463
Accuracy: 15.05%
Best model updated

Epoch 5
Loss: 3.7595
Accuracy: 17.25%
Best model updated

Epoch 6
Loss: 3.6919
Accuracy: 19.12%
Best model updated

Epoch 7
Loss: 3.6381
Accuracy: 20.38%
Best model updated

Epoch 8
Loss: 3.5950
Accuracy: 21.47%
Best model updated

Epoch 9
Loss: 3.5575
Accuracy: 21.67%
Best model updated

Epoch 10
Loss: 3.5288
Accuracy: 22.06%
Best model updated

Epoch 11
Loss: 3.4972
Accuracy: 23.45%
Best model updated

Epoch 12
Loss: 3.4728
Accuracy: 23.81%
Best model updated

Epoch 13
Loss: 3.4485
Accuracy: 24.07%
Best model updated

Epoch 14
Loss: 3.4261
Accuracy: 25.13%
Best model updated

Epoch 15
Loss: 3.4083
Accuracy: 24.89%

Epoch 16
Loss: 3.3922
Accuracy: 25.67%
Best model updated

Epoch 17
Loss: 3.3763
Accuracy: 25.40%

Epoch 18
Loss: 3.3599
Accuracy: 26.28%
Best 

**Multi-exit Transformer**

In [ ]:
import torch
import torch.nn as nn

class MultiExitTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.embed_dim = 64
        self.num_layers = 3

        self.patch_embed = nn.Conv2d(3, self.embed_dim, kernel_size=4, stride=4)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 65, self.embed_dim))

        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=self.embed_dim,
                nhead=4,
                dim_feedforward=128,
                dropout=0.1,
                activation='gelu',
                batch_first=True
            )
            for _ in range(self.num_layers)
        ])

        self.exit_heads = nn.ModuleList([
            nn.Linear(self.embed_dim, 100)
            for _ in range(self.num_layers)
        ])

        self.norm = nn.LayerNorm(self.embed_dim)

    def forward(self, x, threshold=None):
        B = x.size(0)

        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls, x), dim=1)

        x = x + self.pos_embed

        exits = []

        for i, layer in enumerate(self.layers):
            x = layer(x)

            cls_token = self.norm(x[:, 0])
            logits = self.exit_heads[i](cls_token)

            exits.append(logits)

            if threshold is not None:
                energy = -torch.logsumexp(logits, dim=1)

                if (energy < threshold).all():
                    return logits, i + 1

        return exits, self.num_layers

In [ ]:
model = MultiExitTransformer().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

In [ ]:
def multi_exit_loss(outputs, labels):
    loss = 0
    weights = [0.3, 0.3, 0.4]

    for i, out in enumerate(outputs):
        loss += weights[i] * criterion(out, labels)

    return loss

In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs, _ = model(images, threshold=None)

        loss = multi_exit_loss(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(trainloader)

In [ ]:
def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs, _ = model(images, threshold=None)

            final_logits = outputs[-1]

            _, predicted = torch.max(final_logits, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

**Training Multi-exit Transformer**

In [ ]:
EPOCHS = 50
PATIENCE = 10

best_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    loss = train_one_epoch()
    acc = evaluate()

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy (final exit): {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        patience_counter = 0

        torch.save(model.state_dict(), "best_multi_exit.pth")
        print("Best model updated")

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered")
        print(f"Best Accuracy: {best_acc:.2f}%")
        break


Epoch 1
Loss: 4.4312
Accuracy (final exit): 6.49%
Best model updated

Epoch 2
Loss: 4.0279
Accuracy (final exit): 11.24%
Best model updated

Epoch 3
Loss: 3.8262
Accuracy (final exit): 14.52%
Best model updated

Epoch 4
Loss: 3.6987
Accuracy (final exit): 17.08%
Best model updated

Epoch 5
Loss: 3.6004
Accuracy (final exit): 18.52%
Best model updated

Epoch 6
Loss: 3.5173
Accuracy (final exit): 19.67%
Best model updated

Epoch 7
Loss: 3.4542
Accuracy (final exit): 21.27%
Best model updated

Epoch 8
Loss: 3.4031
Accuracy (final exit): 22.19%
Best model updated

Epoch 9
Loss: 3.3603
Accuracy (final exit): 22.11%

Epoch 10
Loss: 3.3272
Accuracy (final exit): 22.81%
Best model updated

Epoch 11
Loss: 3.2944
Accuracy (final exit): 23.48%
Best model updated

Epoch 12
Loss: 3.2660
Accuracy (final exit): 24.22%
Best model updated

Epoch 13
Loss: 3.2421
Accuracy (final exit): 24.33%
Best model updated

Epoch 14
Loss: 3.2179
Accuracy (final exit): 24.96%
Best model updated

Epoch 15
Loss: 3.196

In [ ]:
def evaluate_with_exit(threshold):
    model.eval()

    correct = 0
    total = 0
    exit_layers = []

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs, exit_layer = model(images, threshold=threshold)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            exit_layers.append(exit_layer)

    acc = 100 * correct / total
    avg_exit = sum(exit_layers) / len(exit_layers)

    return acc, avg_exit

In [ ]:
thresholds = [-5, -3, -1]

for t in thresholds:
    acc, avg_exit = evaluate_with_exit(t)

    print(f"\nThreshold: {t}")
    print(f"Accuracy: {acc:.2f}%")
    print(f"Average Exit Layer: {avg_exit:.2f}")


Threshold: -5
Accuracy: 28.56%
Average Exit Layer: 1.65

Threshold: -3
Accuracy: 25.07%
Average Exit Layer: 1.00

Threshold: -1
Accuracy: 25.07%
Average Exit Layer: 1.00


In [ ]:
!pip install thop

In [ ]:
from thop import profile

dummy = torch.randn(1, 3, 32, 32).to(device)

macs, params = profile(model, inputs=(dummy,), verbose=False)

print(f"MACs: {macs / 1e6:.2f} Million")
print(f"Params: {params / 1e6:.2f} Million")

MACs: 3.51 Million
Params: 0.07 Million


In [ ]:
import time

def measure_inference_time():
    model.eval()
    total_time = 0
    total_samples = 0

    with torch.no_grad():
        for images, _ in testloader:
            images = images.to(device)

            start = time.time()
            _ = model(images, threshold=None)
            total_time += time.time() - start

            total_samples += images.size(0)

    return total_time / total_samples

avg_time = measure_inference_time()
print(f"Avg inference time per sample: {avg_time:.6f} sec")

Avg inference time per sample: 0.000022 sec


In [ ]:
def compute_savings(avg_exit_layer, total_layers=3):
    return 1 - (avg_exit_layer / total_layers)

In [ ]:
def evaluate_with_exit(threshold):
    model.eval()

    correct = 0
    total = 0
    exit_layers = []

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs, exit_layer = model(images, threshold=threshold)

            if isinstance(outputs, list):
                logits = outputs[-1]  # final layer
            else:
                logits = outputs      # early exit

            _, predicted = torch.max(logits, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            exit_layers.append(exit_layer)

    acc = 100 * correct / total
    avg_exit = sum(exit_layers) / len(exit_layers)

    return acc, avg_exit

In [ ]:
thresholds = [-6, -5, -4, -3, -2, -1]

results = []

for t in thresholds:
    acc, avg_exit = evaluate_with_exit(t)
    saving = compute_savings(avg_exit)

    results.append((t, acc, avg_exit, saving))

    print(f"\nThreshold: {t}")
    print(f"Accuracy: {acc:.2f}%")
    print(f"Avg Exit Layer: {avg_exit:.2f}")
    print(f"Compute Saved: {saving*100:.2f}%")


Threshold: -6
Accuracy: 32.27%
Avg Exit Layer: 3.00
Compute Saved: 0.00%

Threshold: -5
Accuracy: 28.56%
Avg Exit Layer: 1.65
Compute Saved: 45.01%

Threshold: -4
Accuracy: 25.07%
Avg Exit Layer: 1.00
Compute Saved: 66.67%

Threshold: -3
Accuracy: 25.07%
Avg Exit Layer: 1.00
Compute Saved: 66.67%

Threshold: -2
Accuracy: 25.07%
Avg Exit Layer: 1.00
Compute Saved: 66.67%

Threshold: -1
Accuracy: 25.07%
Avg Exit Layer: 1.00
Compute Saved: 66.67%


In [ ]:
model.load_state_dict(torch.load("best_multi_exit.pth"), strict=False)
model.eval()

MultiExitTransformer(
  (patch_embed): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4))
  (layers): ModuleList(
    (0-2): 3 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (linear1): Linear(in_features=64, out_features=128, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=128, out_features=64, bias=True)
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
    )
  )
  (exit_heads): ModuleList(
    (0-2): 3 x Linear(in_features=64, out_features=100, bias=True)
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)